In [1]:
import os
import optuna
import ipynbname
import pandas as pd
import numpy as np
from Functions.AutoCloud_V4_5 import *
from Functions.TedaOptimize_V4_5 import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.Utils_OPT_R2 import *

FileName = ipynbname.name()

# ---------------- Dados e modelo ----------------
RS_PATH     = r'Dataset\DSIR1n\B1_1.csv'
HI_PATH     = r'Dataset\HI1\Bearing1_1.csv'
PARAMS_PATH = r'Optimization1\TedaRNN_Opt_T2_L46_symmetric_G0P0\mono\Optimization.csv'   # melhor linha = menor MAPE_RUL+MAPE_HI
RHO_TAG     = None        # FileName do OptimizeRho: None -> Optimization\None_symmetric_G0P0\rho

# ---------------- Configuração do experimento ----------------
FEEDBACK = 'symmetric'    # 'symmetric' (gradiente no passo atual) ou 'random' (ablação: RFLO + DFA)
WTAG, WTAP = False, False # configuração principal: soma ponderada dos grânulos (eq. 36)
EOL = 0.3                 # limiar de fim de vida (EoL) do HI
W = 1                     # janela do DSI em cada X_k (nº de amostras do RS; PrepareData/WindowDSI)
LAM = 1.0                 # fator de esquecimento (1.0 = todos os passos com o mesmo peso)

# rho_HI dinâmico: subida de gradiente online em V do HI (cobertura suavizada * especificidade), causal
RHO_DYN, RHO_MIN, RHO_MAX = True, 1e-2, 10
RLS_MODE   = 'pol'        # modelo do erro do HI: 'pol' (polinômio de grau nf_rls) ou 'log'
INTR_STATE = 'carry'      # só afeta rul_bounds='network'

# RUL intervalar (AutoCloud.RULInterv)
RUL_BOUNDS = 'error'      # 'error': predição -/+ rho_rul*ê*g(n) | 'network': análise intervalar da rede
RUL_GROWTH = 'sqrt'      # g(n): 'const' | 'sqrt' | 'lin'
RUL_ECLIP  = 'observed'   # ê nas trajetórias da RUL: 'none' (extrapola) | 'observed' (clip na faixa do HI observado) | 'last' (ê(h_k))
RUL_FRAC   = False        # True: RUL prevista e RUL real fracionárias (interpolação do cruzamento do EoL)
RHO_RUL    = None         # escala da RUL intervalar; None = usa rho_HI (ou a coluna rho_rul da tabela, se houver)

# OptimizeRho
VTARGET   = 'HI'          # 'HI' (causal) | 'RUL' | 'HI*RUL' (usam a RUL real: oráculo)
RHO_RULS  = None          # faixa [min, max] do rho_rul (só com VTARGET 'RUL' ou 'HI*RUL'); None = não otimiza
N_TRIALS  = 200
OPT_RESET = True          # True apaga o estudo anterior de mesmo nome antes de rodar

RHO_CSV = os.path.join('Optimization', RunTag(RHO_TAG, FEEDBACK, WTAG, WTAP), 'rho', 'Optimization.csv')


def LoadRho(src=RHO_CSV, row=0):
    '''Parâmetros do rho (linha `row` da tabela do OptimizeRho, ordenada por V) como dict, lidos pelo nome da coluna.
    A tabela é sobrescrita pelo último OptimizeRho deste RHO_TAG (qualquer Vtarget/rls_mode): confere o rls_mode.'''
    df = pd.read_csv(src) if isinstance(src, str) else src
    p = df.iloc[row].to_dict()
    if ('nf_rls' in p) != (RLS_MODE == 'pol'):
        raise ValueError(f"A tabela {src if isinstance(src, str) else 'dfRho'} não corresponde a RLS_MODE='{RLS_MODE}' "
                         f"(colunas: {list(df.columns)}). Rode o OptimizeRho com este RLS_MODE.")
    return p


def Simulate(p):
    '''Simulação completa (granulação + HI + RUL intervalares) com os parâmetros de rho p (dict de LoadRho).'''
    rho_rul = p['rho_rul'] if RHO_RUL is None and 'rho_rul' in p else RHO_RUL
    teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,ηS=[N1,N2,N3],act=act,tau=τ,
                     eol=EOL,ref=ref,wtaG=WTAG,wtaP=WTAP,feedback=FEEDBACK,mO=mO,
                     rho=p['rho'],lam=LAM,intr_state=INTR_STATE,
                     rho_dyn=RHO_DYN,rho_eta=p['eta'],rho_T=p['T'],rho_beta=p['beta'],rho_min=RHO_MIN,rho_max=RHO_MAX,
                     rls_mode=RLS_MODE,nf_rls=int(p['nf_rls']) if RLS_MODE == 'pol' else None,ff_rls=p['ff_rls'],dt_rls=p['dt_rls'],
                     rul_bounds=RUL_BOUNDS,rul_growth=RUL_GROWTH,rho_rul=rho_rul,rul_eclip=RUL_ECLIP,rul_frac=RUL_FRAC)
    teda.SimGranularRUL(J,L,X,Y,Z)
    return teda


def Report(teda):
    Vr, cr, sr = GranV(teda.rulR, teda.rulL, teda.rulU, np.max(teda.rulR) - np.min(teda.rulR))
    print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | "
          f"rho_HI {teda.rho_used[0]:.3f} -> {teda.rho_used[-1]:.3f}")
    print(f"RUL: cov={cr:.3f} | sp={sr:.3f} | V={Vr:.4f} (Range = max-min da RUL real: oráculo) | "
          f"rulU truncada em L: {int(teda.rulCapU.sum())}/{len(teda.rulU)} | rulU-rulL<1: {int(np.sum(teda.rulU-teda.rulL<1))}")
    print(f"WAPE-RUL: {teda.wape_RUL:.5f} | WAPE-HI: {teda.wape_HI:.5f}")

c:\Users\Usuario\Desktop\Python\Batteries_Prognostics\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
# Dados, modelo (melhor linha de PARAMS_PATH) e simulação com o rho armazenado em RHO_CSV
RS = pd.read_csv(RS_PATH)
HI = pd.read_csv(HI_PATH)
sig = HI['PC1'].values
dfP = pd.read_csv(PARAMS_PATH)
params = dfP.sort_values(dfP.columns[0], ascending=True).iloc[0, 1:].values

m,nI,nR,nO,mO,N1,N2,N3,τ,act = ParseParams(params)
X, Y, Z, kX = PrepareData(RS, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX, frac=RUL_FRAC)   # avisa se o fluxo não alcança o EoL; EoL fracionário com RUL_FRAC
print(f'params = {[m,nI,nR,nO,mO,N1,N2,N3,τ,act]} | J = {J} | L = {L} | ref = {ref}')

pRho = LoadRho()
print('rho:', {k: round(float(v), 6) for k, v in pRho.items()})
teda = Simulate(pRho)
Report(teda)
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')



params = [1.9, 4, [45, 53, 65, 46, 70], 8, 0, np.int64(1), np.int64(5), np.int64(9), np.int64(14), 0] | J = 110 | L = 120 | ref = 112
rho: {'V': 0.800119, 'rho': 0.464703, 'eta': 0.819817, 'T': 0.19566, 'beta': 0.29385, 'nf_rls': 3.0, 'ff_rls': 0.870012, 'dt_rls': 974.849019}
HI : cov=0.836 | sp=0.884 | V=média(cov*sp)=0.7820 | Vsum=86.0227 | rho_HI 0.465 -> 2.439
RUL: cov=0.727 | sp=0.716 | V=0.4521 (Range = max-min da RUL real: oráculo) | rulU truncada em L: 31/110 | rulU-rulL<1: 6
WAPE-RUL: 0.08249 | WAPE-HI: 0.00863


In [ ]:
# Otimização do rho_HI dinâmico (valor inicial, eta, T, beta) e do modelo RLS do erro, para o modelo fixo acima.
# Com VTARGET='HI' as opções da RUL não alteram o objetivo: passam com o padrão para manter o nome do estudo
# (rho_HI_<rls>_err<growth>_<id>) e não abrir um estudo novo com o mesmo objetivo.
rul_opts = {} if VTARGET == 'HI' else dict(rho_rulS=RHO_RULS, rul_eclip=RUL_ECLIP, rul_frac=RUL_FRAC)
dfRho = OptimizeRho(RS=RS, HI=HI, FileName=RHO_TAG, params=params, Vtarget=VTARGET, rls_mode=RLS_MODE,
                    rul_bounds=RUL_BOUNDS, rul_growth=RUL_GROWTH, rho_min=RHO_MIN, rho_max=RHO_MAX, intr_state=INTR_STATE,
                    feedback=FEEDBACK, wtaG=WTAG, wtaP=WTAP, eol=EOL, w=W, lam=LAM,
                    n_trials=N_TRIALS, OptReset=OPT_RESET, **rul_opts)
dfRho

In [8]:
# Simulação final com o melhor rho encontrado (dfRho da célula anterior; sem ela, lê RHO_CSV)
pRho = LoadRho(dfRho if 'dfRho' in globals() else RHO_CSV)
print('rho:', {k: round(float(v), 6) for k, v in pRho.items()})
teda = Simulate(pRho)
Report(teda)
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

rho: {'V': 0.800119, 'rho': 0.464703, 'eta': 0.819817, 'T': 0.19566, 'beta': 0.29385, 'nf_rls': 3.0, 'ff_rls': 0.870012, 'dt_rls': 974.849019}
HI : cov=0.855 | sp=0.888 | V=média(cov*sp)=0.8001 | Vsum=88.0131 | rho_HI 0.465 -> 2.404
RUL: cov=0.809 | sp=0.704 | V=0.5160 (Range = max-min da RUL real: oráculo) | rulU truncada em L: 34/110 | rulU-rulL<1: 9
WAPE-RUL: 0.04307 | WAPE-HI: 0.00738
